## **01_wte: The Word-Vector Dictionary: Token Embeddings**
#### The Goal: Mapping Words to a "Semantic Space"

Our goal is to create a learnable representation for each token. We want to map each token ID to a vector—a point in a high-dimensional space. The key idea is that the *location* of these points should be meaningful.

**Analogy: A Color Space.** Imagine we want to represent colors.
*   **Bad way (Categorical ID):** `{"red": 1, "orange": 2, "blue": 8}`. The number `8` for "blue" has no relation to `1` for "red".
*   **Good way (Vector/Coordinate):** Represent colors in a 2D space where the `x-axis` is "redness" and the `y-axis` is "blueness".
    *   `red` might be `(0.9, 0.1)`
    *   `orange` might be `(0.8, 0.2)` (close to red!)
    *   `blue` might be `(0.1, 0.9)` (far from red!)

Now, the distance between points is meaningful! We are going to do the same thing, but for words, in a space with `n_embd` (e.g., 768) dimensions.

#### The Mechanism: A Learnable Coordinate Book

The `nn.Embedding` layer is this coordinate book. It is a simple lookup table stored as a single weight matrix. Let's build it and inspect its contents.

In [13]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# A tiny config for our example
vocab_size = 10
n_embd = 3 # The number of dimensions in our "semantic space"

# The layer is our coordinate book
token_embedding_table = nn.Embedding(vocab_size, n_embd)

# The book itself is the `.weight` attribute. Each row is a word's coordinate.
print("Shape of our coordinate book:", token_embedding_table.weight.shape)
print("Content of the book (initially random coordinates):")
print(token_embedding_table.weight)

Shape of our coordinate book: torch.Size([10, 3])
Content of the book (initially random coordinates):
Parameter containing:
tensor([[ 1.9269,  1.4873,  0.9007],
        [-2.1055,  0.6784, -1.2345],
        [-0.0431, -1.6047, -0.7521],
        [ 1.6487, -0.3925, -1.4036],
        [-0.7279, -0.5594, -2.3169],
        [-0.2168, -1.3847, -0.8712],
        [-0.2234,  1.7174,  0.3189],
        [-0.4245, -0.8286,  0.3309],
        [-1.5576,  0.9956, -0.8798],
        [-0.6011, -1.2742,  2.1228]], requires_grad=True)


`requires_grad=True - pytorch mechanism for model learning`

In [14]:
B, T = 1, 3 # Batch, Time
# idx = torch.randint(0, vocab_size, (B, T)) # A batch of token ID sequences
idx = torch.tensor([[2, 5, 1]])
# --- The Lookup ---
# For each ID in `idx`, we retrieve its coordinate vector from the table
tok_emb = token_embedding_table(idx)

print(idx)
print("Input IDs shape:", idx.shape)
print("Output Vectors (Coordinates) shape:", tok_emb.shape)
print(tok_emb)

tensor([[2, 5, 1]])
Input IDs shape: torch.Size([1, 3])
Output Vectors (Coordinates) shape: torch.Size([1, 3, 3])
tensor([[[-0.0431, -1.6047, -0.7521],
         [-0.2168, -1.3847, -0.8712],
         [-2.1055,  0.6784, -1.2345]]], grad_fn=<EmbeddingBackward0>)


We successfully transform our data:  
Input: `Meaningless integers`  
*to*  
Output: `Rich 3D Embedding vectors`  
token 2: [-0.0431, -1.6047, -0.7521]  
token 5: [-0.2168, -1.3847, -0.8712]  
token 1: [-2.1055,  0.6784, -1.2345]  

#### **So what have we gained?**  
We now have a **context-free representation** of each word.  
During training, the model learns to place words with similar meanings near each other.  
This is what enables the famous analogy:  
`vector('King') - vector('Man') + vector('Woman') ≈ vector('Queen')`.  
The vector for 'King' captures a concept of "maleness" and "royalty" that can be manipulated mathematically.

#### **But what is the insufficiency?**  
This representation has one massive flaw: it is **static and context-free**. The vector for the word "bank" is identical in these two sentences:
1.  "I sat on the river **bank**."
2.  "I withdrew money from the **bank**."